# 06 · Ablasi Fitur, Skala Data, dan Strategi Target
**Bab III Tabel 3.5.** Semua run memakai hiperparameter terbaik dari notebook 05.

- **A1–A3** (RM2): embedding saja, + rating, + verified_purchase. A4 = U.
- **E1–E5** (RM2): model U pada 100.000 sampai 1.000.000 ulasan latih. Data validasi dan uji tidak berubah.
- **T1–T3** (RM3): target mentah, *age-matched pairs*, dan target terkoreksi usia.

Eksperimen yang dijalankan mengikuti `cfg.run_priority`.

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
from hr import experiment as X
b = X.load_bundle(cfg)
MS, TG = cfg.main_scale, cfg.main_target
best_hp = json.loads(cfg.path("best_hparams.json").read_text())
todo = cfg.experiments_to_run()
best_hp

## 1. Ablasi fitur

In [ ]:
for code in ["A1", "A2", "A3"]:
    if code in todo:
        for seed in cfg.seeds:
            X.run_torch(b, code, scale=MS, target=TG, seed=seed, **best_hp, verbose=False)

## 2. Skala data latih

In [ ]:
for sc in cfg.scales_to_run():
    for seed in cfg.seeds:
        r = X.run_torch(b, "U", scale=sc, target=TG, seed=seed, **best_hp, verbose=False)
        print(sc, seed, r["train_reviews"], round(r["best_val_ndcg@10"], 4), f'{r["minutes"]:.1f} menit')

## 3. Strategi target
**T2** memakai target mentah tetapi pasangan hanya dari ulasan berusia mirip. Batas selisih usia ditala pada data validasi.
**T3** memakai target terkoreksi usia, dan *early stopping* memakai relevansi terkoreksi.

In [ ]:
tg_todo = cfg.targets_to_run()
if "T2" in tg_todo:
    t2 = []
    for w in cfg.age_window_grid:
        r = X.run_torch(b, "U", scale=MS, target="T2", seed=cfg.seed, **best_hp, age_window=w,
                        extra=f"tune_w{w}", verbose=False)
        t2.append({"age_window": w, "val_ndcg@10": r["best_val_ndcg@10"]})
    t2 = pd.DataFrame(t2); display(t2)
    w_best = float(t2.loc[t2["val_ndcg@10"].idxmax(), "age_window"])
    for seed in cfg.seeds:
        X.run_torch(b, "U", scale=MS, target="T2", seed=seed, **best_hp, age_window=w_best, verbose=False)
if "T3" in tg_todo:
    for seed in cfg.seeds:
        X.run_torch(b, "U", scale=MS, target="T3", seed=seed, **best_hp, verbose=False)

## 4. Ringkasan validasi

In [ ]:
tab = X.validation_table(cfg)
tab.to_csv(cfg.path("tables", "validasi_semua.csv"), index=False); tab

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()